# 03 — Transfer Learning con ResNet-18 preentrenada
**TP Semana 6 — Arquitecturas CNN — Tarea 3 (6 pts)**

Usaremos **ResNet-18** preentrenada en ImageNet y compararemos **tres estrategias** sobre
el dataset Blood Cell:

| Estrategia | Capas entrenables |
|---|---|
| **Feature Extraction** | Solo FC final |
| **Fine-tuning parcial** | Últimos bloques (`layer3`, `layer4`) + FC |
| **Fine-tuning total** | Todas las capas (con LR muy pequeño) |

Comparamos las 3 entre sí y contra el mejor modelo *from scratch* de la Tarea 1.


## 1. Setup

In [ ]:
import os, random, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
DATA_ROOT = Path("/content/data")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("./data")

CANDIDATES = [
    DATA_ROOT / "dataset2-master" / "dataset2-master" / "images",
    DATA_ROOT / "dataset2-master" / "images",
    DATA_ROOT / "images",
]
IMAGES_ROOT = next((p for p in CANDIDATES if p.exists()), None)
assert IMAGES_ROOT is not None, "Dataset no encontrado."

TRAIN_DIR = IMAGES_ROOT / "TRAIN"
TEST_DIR  = IMAGES_ROOT / "TEST"


## 2. DataLoaders a 224×224 con normalización de ImageNet

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# Stats de ImageNet — obligatorios para usar pesos preentrenados
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_full = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
test_set   = datasets.ImageFolder(TEST_DIR,  transform=test_tf)

n_total = len(train_full)
n_val   = int(0.1 * n_total)
n_train = n_total - n_val
train_set, val_set = torch.utils.data.random_split(
    train_full, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

CLASSES = train_full.classes
NUM_CLASSES = len(CLASSES)
print("Clases:", CLASSES)
print(f"train={len(train_set)} | val={len(val_set)} | test={len(test_set)}")


## 3. Builders para las tres estrategias

In [ ]:
def build_resnet18(strategy: str, num_classes: int = 4):
    """Devuelve (modelo, params_a_entrenar) según estrategia.

    strategy ∈ {'feature_extraction', 'fine_tune_partial', 'fine_tune_full'}
    """
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)

    # Reemplazar la capa FC final (ImageNet=1000 -> 4 clases del dataset)
    in_feat = model.fc.in_features
    model.fc = nn.Linear(in_feat, num_classes)

    if strategy == "feature_extraction":
        # Congelar todo excepto fc
        for p in model.parameters():
            p.requires_grad = False
        for p in model.fc.parameters():
            p.requires_grad = True

    elif strategy == "fine_tune_partial":
        # Congelar conv1, bn1, layer1, layer2 (los "primeros 2 bloques")
        # Entrenar layer3, layer4 y fc
        for p in model.parameters():
            p.requires_grad = False
        for name, p in model.named_parameters():
            if name.startswith(("layer3", "layer4", "fc")):
                p.requires_grad = True

    elif strategy == "fine_tune_full":
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Estrategia desconocida: {strategy}")

    params_to_train = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params_to_train)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"[{strategy}] parámetros entrenables: {n_trainable:,} / {n_total:,} "
          f"({100*n_trainable/n_total:.1f}%)")

    return model, params_to_train


# Sanity check
_m, _ = build_resnet18("feature_extraction")
print(_m(torch.randn(2, 3, 224, 224)).shape)


## 4. Loops de entrenamiento y evaluación

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, n, correct = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return loss_sum / n, correct / n


def train_transfer(model, params_to_train, lr, epochs, tag):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(params_to_train, lr=lr)

    history = {"epoch": [], "train_loss": [], "train_acc": [],
               "val_loss": [], "val_acc": [], "time_s": []}

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        loss_sum, n, correct = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * x.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            n += x.size(0)

        train_loss = loss_sum / n
        train_acc  = correct  / n
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        dt = time.time() - t0

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["time_s"].append(dt)

        print(f"[{tag}] ep {epoch:02d}/{epochs}  "
              f"train_loss={train_loss:.4f} acc={train_acc:.3f}  "
              f"val_loss={val_loss:.4f} acc={val_acc:.3f}  ({dt:.1f}s)")
    return history


## 5. Entrenamiento de las tres estrategias

In [ ]:
EPOCHS = 8

configs = [
    # (estrategia,            lr,      epochs)
    ("feature_extraction",    1e-3,    EPOCHS),
    ("fine_tune_partial",     5e-4,    EPOCHS),
    ("fine_tune_full",        1e-4,    EPOCHS),   # LR muy pequeño
]

models_tl = {}
histories_tl = {}

for strategy, lr, ep in configs:
    print(f"\n========= {strategy.upper()} (lr={lr}) =========")
    model, params = build_resnet18(strategy, num_classes=NUM_CLASSES)
    hist = train_transfer(model, params, lr=lr, epochs=ep, tag=strategy)
    models_tl[strategy] = model
    histories_tl[strategy] = hist


## 6. Comparación final en TEST

In [ ]:
criterion = nn.CrossEntropyLoss()
rows = []
for strategy, model in models_tl.items():
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    rows.append({
        "estrategia": strategy,
        "params entrenables": n_trainable,
        "best val_acc": max(histories_tl[strategy]["val_acc"]),
        "test_acc": test_acc,
        "test_loss": test_loss,
        "tiempo/epoch (s)": float(np.mean(histories_tl[strategy]["time_s"])),
    })

summary_tl = pd.DataFrame(rows).set_index("estrategia")
summary_tl.to_csv("resumen_tarea3.csv")
summary_tl


In [ ]:
# Curvas de las 3 estrategias
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = {"feature_extraction": "#3b82f6",
          "fine_tune_partial":  "#10b981",
          "fine_tune_full":     "#ef4444"}

for strategy, h in histories_tl.items():
    c = colors[strategy]
    axes[0].plot(h["epoch"], h["train_loss"], "--", color=c, label=f"{strategy} train")
    axes[0].plot(h["epoch"], h["val_loss"], "-",   color=c, label=f"{strategy} val")
    axes[1].plot(h["epoch"], h["train_acc"], "--", color=c, label=f"{strategy} train")
    axes[1].plot(h["epoch"], h["val_acc"], "-",    color=c, label=f"{strategy} val")

axes[0].set_title("Transfer Learning — pérdida"); axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_title("Transfer Learning — accuracy"); axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("curvas_transfer.png", dpi=120, bbox_inches="tight")
plt.show()


## 7. Matriz de confusión y reporte del mejor modelo

In [ ]:
best_strategy = summary_tl["test_acc"].idxmax()
best_model = models_tl[best_strategy]
print(f"Mejor estrategia: {best_strategy}  (test_acc={summary_tl.loc[best_strategy, 'test_acc']:.3f})")

best_model.eval()
all_preds, all_y = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        all_preds.append(best_model(x).argmax(1).cpu().numpy())
        all_y.append(y.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_y)

print("\n--- Classification report ---")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))


In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=30, ha="right"); ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicción"); ax.set_ylabel("real")
ax.set_title(f"Matriz de confusión — {best_strategy}")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig("matriz_confusion.png", dpi=120, bbox_inches="tight")
plt.show()


## 8. Comparación contra el mejor modelo *from scratch* (Tarea 1)

> Carga el archivo `resumen_tarea1.csv` generado por el notebook 02 y compáralo con
> los resultados de Transfer Learning.


In [ ]:
try:
    summary_t1 = pd.read_csv("resumen_tarea1.csv", index_col=0)
    best_scratch = summary_t1["test_acc"].idxmax()
    best_scratch_acc = summary_t1.loc[best_scratch, "test_acc"]
    print(f"Mejor modelo from scratch: {best_scratch}  (test_acc={best_scratch_acc:.3f})")
    print(f"Mejor modelo transfer    : {best_strategy}  (test_acc={summary_tl.loc[best_strategy, 'test_acc']:.3f})")
except FileNotFoundError:
    print("Ejecuta primero 02_lenet_vgg.ipynb para generar resumen_tarea1.csv")


## 9. Discusión y recomendación

**Comparación entre estrategias:**
- **Feature Extraction** entrena pocos parámetros (~2 K) y es la opción **más rápida**;
  funciona bien como *baseline* y suele dar accuracies sólidas porque las features de
  ImageNet ya capturan texturas y bordes útiles.
- **Fine-tuning parcial** suele dar el **mejor balance**: descongelar `layer3`/`layer4`
  permite que la red especialice los filtros de alto nivel al dominio médico, mientras
  conserva las features de bajo nivel (bordes, gradientes de color) que ya están bien
  aprendidas.
- **Fine-tuning total** con LR muy pequeño puede igualar o superar al parcial, pero es el
  más caro computacionalmente y el más sensible a *overfitting* si el dataset es
  pequeño.

**Recomendación para datos médicos limitados:**

En contextos médicos típicos —dataset relativamente pequeño, imágenes con patrones de
tinción / morfología muy específicos, y necesidad de robustez— recomendamos **fine-tuning
parcial**:

1. Las primeras capas (bordes, color, textura básica) ya generalizan bien y no requieren
   re-entrenamiento; congelarlas **reduce el riesgo de sobreajuste**.
2. Las capas profundas (`layer3`, `layer4`) son las que codifican conceptos abstractos
   específicos de ImageNet (objetos cotidianos); re-entrenarlas las adapta a las
   estructuras celulares.
3. El número de parámetros entrenables es manejable, lo que permite **menos épocas** y
   reduce el costo computacional frente a fine-tuning total.
4. Empíricamente suele alcanzar la mejor accuracy de validación / test con el menor
   tiempo de entrenamiento, y es el patrón estándar recomendado en la literatura de
   imágenes médicas cuando se tienen unos miles de imágenes por clase.

Si el dataset fuese mucho más pequeño (<200 imágenes/clase), **Feature Extraction** sería
preferible para minimizar overfitting. Si fuera mucho mayor (>10⁵ imágenes), **fine-tuning
total** podría justificar su costo.
